# CLAP API smoke test
Try multiple API patterns to find the working one for transformers 새버전.


In [ ]:
import sys, traceback
import numpy as np
import torch
import transformers

print(f"transformers: {transformers.__version__}")
print(f"torch:        {torch.__version__}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from transformers import ClapModel, ClapProcessor
CLAP_NAME = "laion/clap-htsat-unfused"
print(f"\nLoading {CLAP_NAME}...")
proc = ClapProcessor.from_pretrained(CLAP_NAME)
model = ClapModel.from_pretrained(CLAP_NAME).to(DEVICE).eval()
SR = proc.feature_extractor.sampling_rate
print(f"SR = {SR}")

# Dummy audio batch
batch = [np.random.randn(SR * 5).astype(np.float32) for _ in range(3)]
inputs = proc(audio=batch, sampling_rate=SR, return_tensors="pt", padding=True)
inputs = {k: v.to(DEVICE) for k, v in inputs.items() if hasattr(v, 'to')}
print(f"\ninputs keys: {list(inputs.keys())}")
for k, v in inputs.items():
    if hasattr(v, "shape"):
        print(f"  {k}: shape={tuple(v.shape)}, dtype={v.dtype}")

# Method 1: get_audio_features
print("\n--- Method 1: model.get_audio_features(**inputs) ---")
with torch.no_grad():
    try:
        out = model.get_audio_features(**inputs)
        t = type(out).__name__
        print(f"  type: {t}")
        if torch.is_tensor(out):
            print(f"  RESULT: tensor shape={tuple(out.shape)}  ✅")
        else:
            attrs = [a for a in dir(out) if not a.startswith("_")][:25]
            print(f"  attrs: {attrs}")
            for attr in ("audio_embeds", "pooler_output", "last_hidden_state"):
                if hasattr(out, attr):
                    v = getattr(out, attr)
                    if torch.is_tensor(v):
                        print(f"  attr {attr}: {tuple(v.shape)}")
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")

# Method 2: full forward with audio_embeds
print("\n--- Method 2: model(**inputs).audio_embeds ---")
with torch.no_grad():
    try:
        full_out = model(**inputs)
        print(f"  type: {type(full_out).__name__}")
        attrs = [a for a in dir(full_out) if not a.startswith("_")][:25]
        print(f"  attrs: {attrs}")
        if hasattr(full_out, "audio_embeds"):
            v = full_out.audio_embeds
            print(f"  audio_embeds: {tuple(v.shape)} ✅")
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")

# Method 3: model.audio_model(...) directly
print("\n--- Method 3: model.audio_model(**audio_inputs) ---")
with torch.no_grad():
    try:
        audio_inputs = {k: v for k, v in inputs.items()
                        if k in ("input_features", "is_longer")}
        audio_out = model.audio_model(**audio_inputs)
        print(f"  type: {type(audio_out).__name__}")
        for attr in ("pooler_output", "last_hidden_state"):
            if hasattr(audio_out, attr):
                v = getattr(audio_out, attr)
                if torch.is_tensor(v):
                    print(f"  {attr}: {tuple(v.shape)} ✅")
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")
